# 06.13 — Sampling layer deterministic at inference

The VAE's `SamplingLayer` ([06.2](06.2_vae_and_the_elbo.ipynb)) does *two different things* depending on a single boolean. In **training** it draws a random reparameterized sample — that stochasticity is what regularizes the latent space and enables the KL term. In **evaluation** it returns the mean `mu` deterministically — so predictions are reproducible and represent the model's single best estimate (Critical Note #35). The switch is the module's `self.training` flag, flipped by `model.train()` / `model.eval()`. Forgetting to call `eval()` is one of the most common — and most silent — PyTorch bugs, and this notebook is about getting it right.

This notebook covers:

* The `self.training` flag and how `train()` / `eval()` flip it (recursively).
* Stochastic sample (train) vs deterministic mean (eval).
* Why inference must be deterministic — reproducibility and best-estimate.
* The train/eval-sensitive layer family: Dropout, BatchNorm, and this.

**Prerequisites:** [06.2 VAE + ELBO](06.2_vae_and_the_elbo.ipynb), [05.7 batch-norm state](../05_training_loop/05.7_batch_norm_state.ipynb).

## Section 1 — What MATLAB does

MATLAB layers implement separate training and inference paths. `cgg_samplingLayer` samples in `forward` (training) and returns the mean in `predict` (inference):

```matlab
function Z = forward(layer, X)      % TRAINING path
    [mu, logvar] = split(X);
    Z = mu + randn(size(mu)) .* exp(0.5 * logvar);   % reparameterized SAMPLE
end
function Z = predict(layer, X)      % INFERENCE path
    [mu, ~] = split(X);
    Z = mu;                          % deterministic MEAN
end
```

`trainnet` calls `forward`; `predict`/`minibatchpredict` call `predict`. PyTorch collapses these into one `forward` method that branches on `self.training` — the port reproduces the exact same two behaviors (Critical Note #35).

## Section 2 — The Python concepts you need

### 2.1 — The `self.training` flag

Every `nn.Module` carries a boolean `self.training`. `model.train()` sets it to `True` on the module *and all its submodules*; `model.eval()` sets it to `False` everywhere. It's a recursive switch — flip it on the top-level model and every layer inside sees it:

In [ ]:
import torch
import torch.nn as nn
from neural_data_decoding.models.layers.sampling import SamplingLayer

model = nn.Sequential(nn.Linear(6, 6), SamplingLayer())
model.train()
print("after model.train() — SamplingLayer.training =", model[1].training)
model.eval()
print("after model.eval()  — SamplingLayer.training =", model[1].training)
print("→ one call on the parent flips the flag on every submodule, recursively.")

### 2.2 — Two behaviors, one layer

In [ ]:
torch.manual_seed(0)
s = SamplingLayer()
stats = torch.randn(2, 6)          # [mu | logvar] along the channel axis, latent=3

s.train()
z1, mu, _ = s(stats)
z2, _, _ = s(stats)
print("TRAIN — two forwards equal?", torch.equal(z1, z2), " → stochastic (fresh noise each call)")

s.eval()
z3, mu, _ = s(stats)
z4, _, _ = s(stats)
print("EVAL  — two forwards equal?", torch.equal(z3, z4), " → deterministic")
print("EVAL  — z == mu?", torch.equal(z3, mu), " → returns the mean, no noise")

In `train()` the layer computes `z = mu + eps · exp(0.5·logvar)` with a fresh `eps ~ N(0,1)` every call — so two forwards of the *same* input give *different* `z`. In `eval()` it returns `z = mu` — the same input always gives the same `z`. One layer, two regimes, switched by the flag.

### 2.3 — Why inference must be deterministic

Two reasons the mean is right for inference:

* **Reproducibility.** A decoder should give the *same* answer for the same trial every time you run it. If the sampling layer kept drawing random `eps` at inference, your predictions — and your reported accuracy — would jitter from run to run ([05.5](../05_training_loop/05.5_checkpoint_resume_state_machine.ipynb)'s reproducibility concerns). Downstream analysis ("which trials did the model get right?") needs a fixed answer.
* **Best estimate.** `mu` is the *mode* of the latent Gaussian `N(mu, σ²)` — the single most probable latent. At inference you want the model's best guess, not a random draw from its uncertainty. Sampling is a *training* device (it forces the latent space to be smooth and the KL term to be meaningful, [06.2 §2.4](06.2_vae_and_the_elbo.ipynb)); at test time you cash in that smoothness by reading off the center.

### 2.4 — The stochastic cloud collapses to a point

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(1)
stats2d = torch.tensor([[1.5, -0.5, 0.4, 0.4]])    # mu=(1.5,-0.5), logvar=(0.4,0.4), latent=2
s = SamplingLayer()

s.train()
samples = torch.stack([s(stats2d)[0][0] for _ in range(400)])   # 400 training draws
s.eval()
point = s(stats2d)[0][0]                                          # the single eval output

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(samples[:, 0], samples[:, 1], s=12, alpha=0.25, color="#5588cc", label="train: 400 samples")
ax.scatter([point[0]], [point[1]], s=260, color="crimson", marker="*", edgecolor="black",
           zorder=5, label="eval: z = mu (deterministic)")
ax.set_xlabel("latent dim 0"); ax.set_ylabel("latent dim 1")
ax.set_title("Training samples scatter around mu; inference returns mu itself"); ax.legend()
plt.tight_layout(); plt.show()
print("The training cloud (blue) is the noise that regularizes; inference reads the center (red star).")

The blue cloud is the stochasticity that shapes the latent space during training; the red star is what inference returns — the center of that cloud. Same layer, same weights; the flag decides whether you get a draw from the cloud or its center.

### 2.5 — A whole family of train/eval-sensitive layers

`SamplingLayer` isn't special in *being* mode-dependent — it joins a family:

| Layer | `train()` behavior | `eval()` behavior |
|---|---|---|
| **Dropout** | randomly zeros activations | passes everything through |
| **BatchNorm** ([05.7](../05_training_loop/05.7_batch_norm_state.ipynb)) | normalizes by *batch* stats, updates running stats | normalizes by *running* stats |
| **SamplingLayer** | draws a random latent | returns the mean |

All three do something *stochastic or batch-dependent* in training and something *fixed* at inference. That's exactly why `model.eval()` exists — it's one switch that puts every one of these into inference mode at once. Forget it, and Dropout is still dropping, BatchNorm is still using batch stats, and the VAE is still sampling — during what's supposed to be deterministic evaluation.

## Section 3 — The `neural_data_decoding` implementation

The `SamplingLayer.forward` branch — the entire train/eval switch is five lines:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/layers/sampling.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "if self.training:" in line)
for k in range(i - 3, i + 6):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `if self.training:` — the branch reads the flag PyTorch maintains; the layer never sets it itself (that's `train()`/`eval()`'s job).
* Train branch: `eps = torch.randn_like(mu); z = mu + eps * torch.exp(0.5 * logvar)` — the reparameterization trick ([06.2 §2.2](06.2_vae_and_the_elbo.ipynb)), noise scaled by the standard deviation `exp(0.5·logvar)`.
* Eval branch: `z = mu` with the `# deterministic at inference — Critical Note #35` comment. The mean, no noise.
* Both branches return `(z, mu, logvar)` — `mu` and `logvar` are returned *regardless* of mode, because the ELBO's KL term ([06.2 §2.4](06.2_vae_and_the_elbo.ipynb)) needs them even though only training uses the sampled `z` differently.
* This is why the Stochastic vs Deterministic encoder placement ([06.3 §2.4](06.3_stochastic_vs_deterministic_placement.ipynb)) *collapses at eval*: both feed the classifier `mu`, because `z == mu` in eval mode.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the classic bug: forgetting `eval()`

Run "inference" without calling `eval()` and show the predictions are non-reproducible; then fix it with `eval()`.

In [ ]:
# Reveal:
torch.manual_seed(0)
s = SamplingLayer()
stats = torch.randn(1, 6)

# BUG: still in train mode (the default) during "inference"
preds_buggy = [round(s(stats)[0][0, 0].item(), 4) for _ in range(4)]
print("WITHOUT eval():", preds_buggy, "← different every call (silent bug!)")

s.eval()
preds_fixed = [round(s(stats)[0][0, 0].item(), 4) for _ in range(4)]
print("WITH eval():   ", preds_fixed, "← identical, reproducible")

### Exercise 4.2 — the flag is recursive AND toggleable

Show that switching a top-level model between `train()` and `eval()` flips a deeply-nested `SamplingLayer`, and that you can go back and forth (e.g., eval for validation mid-training, then back to train).

In [ ]:
# Reveal:
deep = nn.Sequential(nn.Linear(6, 6), nn.Sequential(nn.ReLU(), SamplingLayer()))
nested = deep[1][1]           # the buried SamplingLayer
for mode in ("train", "eval", "train"):
    getattr(deep, mode)()
    print(f"deep.{mode}()  → nested SamplingLayer.training = {nested.training}")
print("→ one toggle reaches any depth; loops that validate mid-epoch flip eval→train freely.")

## Section 5 — Common errors

### Predictions change every time you run inference

The classic bug (Exercise 4.1): `model.eval()` was never called, so the SamplingLayer is still drawing noise (and Dropout still dropping, BatchNorm still using batch stats). Call `model.eval()` before any validation or prediction pass.

### Validation accuracy is noisy / worse than training

Same cause — evaluating in train mode. The sampling noise (and dropout) degrade the eval-time predictions. Wrap validation in `model.eval()` and `torch.no_grad()`.

### Forgot to switch *back* to `train()` after validating

A mid-epoch validation pass calls `eval()`; if you don't call `train()` afterward, the *rest of training* runs deterministically — no sampling, no dropout, no BatchNorm updates. Toggle back (Exercise 4.2). The training loop ([05.1](../05_training_loop/05.1_the_custom_training_loop.ipynb)) must bracket validation with eval→train.

### Expecting different outputs in eval mode

If you *want* stochastic predictions at inference (e.g., MC-dropout uncertainty), `eval()` disables exactly that. That's a deliberate, non-default choice — the standard VAE inference is the deterministic mean.

### `self.training` set by hand

Don't assign `layer.training = False` directly — use `.eval()` / `.train()` so the change propagates to submodules and stays consistent. Hand-setting one layer's flag while the parent thinks it's in train mode is a subtle desync.

## Section 6 — Further reading

- [`src/neural_data_decoding/models/layers/sampling.py`](../../src/neural_data_decoding/models/layers/sampling.py) — the `self.training` branch (Critical Note #35).
- [05.7 batch-norm state](../05_training_loop/05.7_batch_norm_state.ipynb) — the other headline train/eval-sensitive layer.
- [`nn.Module.train` docs](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train) — the recursive flag mechanism.

**Module 06 complete.** You've traced the full loss orchestration: the four components, the VAE/ELBO, stochastic-vs-deterministic placement, EMA prior normalization (twice — [06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb) and [06.12](06.12_ema_prior_normalization_deep_dive.ipynb)), MIL pooling, the confidence heads and their P-controller, grad-side L2, per-batch correction, NaN-masked reconstruction, and the single-total-loss topology. Next up: **[Module 07 — the dynamic curriculum](../07_dynamic_curriculum/)**, where these losses are switched on and off across training according to a schedule.